# Lab 3: Constructing an analysis-ready longitudinal dataset

## Learning objectives

By the end of this lab, you will be able to:

- use documentation to reconstruct and validate a derived MAPI score before analyzing it;
- distinguish missing item values, absent instrument rows, unavailable laboratory results, and documented special codes;
- describe longitudinal measurement-availability patterns without treating them as proof of attrition;
- distinguish observed values from LOCF-constructed analysis values and evaluate how LOCF changes a descriptive result;
- identify the row unit and composite key of linked participant-session tables;
- join self-report and hair toxicology source tables without overwriting them;
- define and audit a valid denominator for a categorical × categorical analysis; and
- interpret self-report and hair toxicology concordance cautiously.

### Central research question

> Among participant-sessions with both a valid hair confirmation and a completed report of cannabis use since the previous session, how often do self-report and hair confirmation agree or differ?

In DSARM 1, Python is a tool for understanding biomedical data. The goal is not merely to produce a merged file. The goal is to construct a documented analysis-ready table without overwriting or misinterpreting the source records.

### Dataset provenance and connection to Exercise 3

The lab files are synthetic instructional data modeled on real ABCD table and variable structures. Participants are approximately 18 years old at session 07, 19 years old at session 08, and 21 years old at session 10. These session labels establish the simulated longitudinal sequence used throughout the labs.

Exercise 3 introduced why values and records may be unavailable, how availability can change an analytic sample, and how available-observation, complete-case, and LOCF decisions differ. Lab 3 applies those principles to longitudinal, ABCD-shaped synthetic tables.

These are not official ABCD observations, official ABCD waves, or national prevalence estimates.

## 1. Load the source tables

We begin with three separate source tables rather than a pre-merged file:

- `su_y_mjprob`, Marijuana Problem Index items and summary score;
- `su_y_sui`, cannabis self-report items; and
- `su_y_hairtox`, hair collection and toxicology results.

Each CSV becomes a pandas **DataFrame**, a rectangular object organized into rows and columns. The meaning of one row can differ across DataFrames, so we must identify the row unit before combining tables.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Paths are relative to this notebook in labs/
sui_path = "../data/labs_01_03/su_y_sui.csv"
hairtox_path = "../data/labs_01_03/su_y_hairtox.csv"
mjprob_path = "../data/labs_01_03/su_y_mjprob.csv"

su_y_sui = pd.read_csv(sui_path)
su_y_hairtox = pd.read_csv(hairtox_path)
su_y_mjprob = pd.read_csv(mjprob_path)

source_table_summary = pd.DataFrame({
    "table": ["su_y_sui", "su_y_hairtox", "su_y_mjprob"],
    "rows": [len(su_y_sui), len(su_y_hairtox), len(su_y_mjprob)],
    "columns": [su_y_sui.shape[1], su_y_hairtox.shape[1], su_y_mjprob.shape[1]],
})

display(source_table_summary)

### Source tables and analysis DataFrames

A **source variable** is retained as supplied in a source table. A **derived variable** is a new field created for a documented analytic purpose. In this lab, we leave the source DataFrames unchanged and place derived fields in new working DataFrames.

An imputed value is a constructed analysis value, not a recovered observation. Any LOCF value created in this lab will be stored in a new column and will never replace an observed source value.

## 2. Reconstruct and validate the MAPI summary score

The Marijuana Problem Index, or MAPI, demonstrates why documentation must come before cleaning, derivation, or longitudinal analysis. We will inspect item availability, apply the official scoring rule, reconstruct the score, validate it, and only then use it to study availability across sessions.

Instrument documentation:  
https://docs.abcdstudy.org/latest/documentation/non_imaging/su.html#marijuana-problems

### 2.1 Inspect item availability and score inputs

The heatmap includes the 18 numbered MAPI item columns. Each row is one participant-session record. Dark cells are observed responses and bright cells are unavailable values.

An empty cell does not explain itself. We must use the score documentation to determine which items belong in the MAPI score and must not infer that an empty item is zero, a refusal, a skipped visit, or another specific state without supporting evidence.

In [ ]:
mapi_item_cols = [
    f"su_y_mjprob_{item_number:03d}"
    for item_number in range(1, 19)
]

mapi_item_cols = [
    column for column in mapi_item_cols
    if column in su_y_mjprob.columns
]

mapi_missing_matrix = (
    su_y_mjprob[mapi_item_cols]
    .isna()
    .astype(int)
    .to_numpy()
)

plt.figure(figsize=(12, 6))
heatmap = plt.imshow(
    mapi_missing_matrix,
    aspect="auto",
    interpolation="nearest",
    cmap="viridis",
)
plt.title("Missingness across MAPI items")
plt.xlabel("MAPI item")
plt.ylabel("Participant-session rows")
plt.xticks(
    ticks=np.arange(len(mapi_item_cols)),
    labels=[column.split("_")[-1] for column in mapi_item_cols],
)
plt.yticks([])

colorbar = plt.colorbar(heatmap)
colorbar.set_ticks([0, 1])
colorbar.set_ticklabels(["Observed", "Missing"])
plt.tight_layout()
plt.show()

mapi_missing_summary = pd.DataFrame({
    "variable": mapi_item_cols,
    "missing_count": su_y_mjprob[mapi_item_cols].isna().sum().to_numpy(),
    "percent_missing": (
        su_y_mjprob[mapi_item_cols]
        .isna()
        .mean()
        .mul(100)
        .round(1)
        .to_numpy()
    ),
})

display(
    mapi_missing_summary.loc[
        mapi_missing_summary["percent_missing"] > 0
    ]
)

### 2.2 Apply the documented scoring rule

The ABCD scoring documentation identifies the variables summarized in `su_y_mjprob_prsum`:

https://software.nbdc-datahub.org/ABCDscores/reference/compute_su_y_mjprob_prsum.html

The documented rule states that:

- the score uses items `001`–`012` and `016`–`018`;
- these 15 items define the score;
- items `013`–`015` are not included;
- no more than 3 active scoring items may be missing; and
- eligible incomplete rows are prorated to the 15-item scale.

Items `013`–`015` must not be coded as zero. A zero is an observed response, while these items are outside this score definition. Their empty cells also do not, by themselves, establish why they were unavailable.

Applying `dropna()` across all 18 numbered columns would remove every row because items `013`–`015` are entirely blank in this synthetic extract. That would discard valid records even though the documented score uses the other 15 items.

### 2.3 Reconstruct and validate before analysis

For each participant-session row, we count observed active items, calculate their mean, and multiply by 15. The reconstructed value is retained only when no more than 3 active items are missing.

The reconstructed score is compared with the stored `su_y_mjprob_prsum`. Validation checks both numeric agreement and agreement about which rows are unavailable. Distributional or longitudinal analysis should occur only after this validation succeeds.

In [ ]:
mapi_scoring_items = [
    "su_y_mjprob_001", "su_y_mjprob_002", "su_y_mjprob_003",
    "su_y_mjprob_004", "su_y_mjprob_005", "su_y_mjprob_006",
    "su_y_mjprob_007", "su_y_mjprob_008", "su_y_mjprob_009",
    "su_y_mjprob_010", "su_y_mjprob_011", "su_y_mjprob_012",
    "su_y_mjprob_016", "su_y_mjprob_017", "su_y_mjprob_018",
]

mapi_work = su_y_mjprob[
    ["participant_id", "session_id"] + mapi_scoring_items
].copy()

mapi_work[mapi_scoring_items] = (
    mapi_work[mapi_scoring_items]
    .apply(pd.to_numeric, errors="coerce")
)

mapi_work["mapi_items_observed"] = (
    mapi_work[mapi_scoring_items].notna().sum(axis=1)
)
mapi_work["mapi_items_missing"] = (
    len(mapi_scoring_items) - mapi_work["mapi_items_observed"]
)

eligible_for_score = mapi_work["mapi_items_missing"] <= 3

mapi_work["mapi_prsum_reconstructed"] = (
    mapi_work[mapi_scoring_items]
    .mean(axis=1)
    .mul(len(mapi_scoring_items))
    .where(eligible_for_score)
)

mapi_validation = mapi_work.merge(
    su_y_mjprob[
        ["participant_id", "session_id", "su_y_mjprob_prsum"]
    ],
    on=["participant_id", "session_id"],
    how="left",
    validate="one_to_one",
)

reconstructed_score = pd.to_numeric(
    mapi_validation["mapi_prsum_reconstructed"],
    errors="coerce",
)
official_score = pd.to_numeric(
    mapi_validation["su_y_mjprob_prsum"],
    errors="coerce",
)

mapi_validation["scores_match"] = np.isclose(
    reconstructed_score.to_numpy(dtype=float),
    official_score.to_numpy(dtype=float),
    atol=1e-8,
    equal_nan=True,
)

print("Participant-session rows checked:", len(mapi_validation))
print("Matching rows:", int(mapi_validation["scores_match"].sum()))
print("Mismatching rows:", int((~mapi_validation["scores_match"]).sum()))

display(
    mapi_validation[
        [
            "participant_id",
            "session_id",
            "mapi_items_observed",
            "mapi_items_missing",
            "mapi_prsum_reconstructed",
            "su_y_mjprob_prsum",
            "scores_match",
        ]
    ].head(10)
)

### Your Turn 1: create the longitudinal availability pattern

In **wide format**, each participant has one row and the three sessions appear in separate columns. This makes the observed/unavailable pattern visible.

We will create an availability pattern using `1` for an observed MAPI score and `0` for an unavailable score. For example:

- `111`: observed at all three sessions;
- `101`: unavailable at session 08 but observed again at session 10; and
- `110`: observed at sessions 07 and 08 but unavailable at session 10.

Only the `101` pattern directly demonstrates an intermittent MAPI gap followed by a later observed MAPI score. Patterns ending in `0` are not sufficient evidence of attrition. A visits/retention source and administration information would be needed to establish whether a visit was expected, completed, or eligible for the instrument. Session-specific observations may also represent different participant subsets, so differences across available observations do not by themselves demonstrate within-person change.

Complete the blank in the next cell to combine each row's three string values into one pattern such as `101`.

**Clue:** Use the string method that joins several strings with nothing placed between them. The code should work across each row because `axis=1`.

**Further learning with Copilot:** Ask: *In pandas, I have three string columns containing `0` or `1`. Explain how `DataFrame.agg()` can combine the values across each row into one string. Explain what `axis=1` means, but do not complete my code.*

In [ ]:
session_order = ["ses-07A", "ses-08A", "ses-10A"]

mapi_wide = mapi_validation.pivot(
    index="participant_id",
    columns="session_id",
    values="mapi_prsum_reconstructed",
).reindex(columns=session_order)

mapi_presence = mapi_wide.notna().astype(int)
mapi_presence["availability_pattern"] = (
    mapi_presence[session_order].astype(str).agg(______, axis=1)
)

pattern_descriptions = {
    "111": "Observed at all three sessions",
    "101": "Session-08 gap followed by an observed session-10 score",
    "110": "Observed at 07 and 08; no observed score at 10",
    "100": "Observed only at session 07",
    "011": "No observed score at 07; observed at 08 and 10",
    "010": "Observed only at session 08",
    "001": "Observed only at session 10",
}

pattern_order = ["111", "101", "110", "100", "011", "010", "001"]

mapi_pattern_counts = (
    mapi_presence["availability_pattern"]
    .value_counts()
    .reindex(pattern_order, fill_value=0)
    .rename_axis("availability_pattern")
    .reset_index(name="participants")
)
mapi_pattern_counts["description"] = (
    mapi_pattern_counts["availability_pattern"]
    .map(pattern_descriptions)
)

display(mapi_wide.head(10))
display(mapi_pattern_counts)

availability_sorted = (
    mapi_presence
    .sort_values("availability_pattern", ascending=False)
)

plt.figure(figsize=(7, 6))
plt.imshow(
    availability_sorted[session_order].to_numpy(),
    aspect="auto",
    interpolation="nearest",
    cmap="Greys",
    vmin=0,
    vmax=1,
)
plt.title("MAPI score availability by participant and session")
plt.xlabel("Session")
plt.ylabel("Participants, sorted by availability pattern")
plt.xticks(np.arange(len(session_order)), session_order)
plt.yticks([])
colorbar = plt.colorbar()
colorbar.set_ticks([0, 1])
colorbar.set_ticklabels(["Unavailable", "Observed"])
plt.tight_layout()
plt.show()

### Prepare for the Canvas reflection: MAPI validation and availability

Using the validation and availability outputs, respond in four or five sentences:

1. How many participant-session rows were checked, how many matched the stored `su_y_mjprob_prsum`, and how many mismatched?
2. Why should MAPI items `013`-`015` remain distinct from observed zeros?
3. Which availability pattern demonstrates an intermittent MAPI gap followed by a later observed score, and how many participants had that pattern?
4. Why do patterns such as `110` and `100` not prove that a participant left the study?
5. What additional source would be needed to distinguish visit nonattendance, instrument ineligibility, and a measure not scheduled by design?

These questions are assessed in Canvas; no additional notebook response is required.

## 3. Compare observed and LOCF session-10 analysis values

Exercise 3 introduced **last observation carried forward (LOCF)** as a form of single imputation. LOCF can increase the number of analysis values without increasing the amount of observed follow-up evidence.

Here we use LOCF as a sensitivity demonstration, not as a recommended correction. We will compare session-10 MAPI summaries within a clearly defined anchor cohort: participants with an observed session-08 MAPI score.

### 3.1 Define the comparison and its assumption

For the session-08 anchor cohort, we compare:

| Handling decision | Session-10 analysis value |
|---|---|
| Observed only | Use an observed session-10 MAPI score; otherwise leave unavailable |
| LOCF from session 08 | Use the observed session-10 score when present; otherwise carry the observed session-08 score forward |

- The anchor cohort contains participants with an observed session-08 MAPI score. Observed-only analysis uses the subset that also has an observed session-10 score; LOCF constructs a session-10 analysis value for the remainder.
- LOCF assumes that each carried participant's session-08 MAPI score remained unchanged through session 10.
- LOCF does not recover the actual session-10 response or explain why that response was unavailable.

### Your Turn 2: construct the LOCF sensitivity value

Complete the one blank in the code below. The new LOCF column must preserve every observed session-10 score and use session 08 only when session 10 is unavailable. The source indicator keeps observed and constructed values distinguishable.

In [ ]:
mapi_08_anchor = mapi_wide.loc[
    mapi_wide["ses-08A"].notna()
].copy()

mapi_08_anchor["mapi_10A_observed"] = (
    mapi_08_anchor["ses-10A"]
)

# TODO: Fill an unavailable session-10 score from the observed session-08 score.
mapi_08_anchor["mapi_10A_locf_from_08A"] = (
    mapi_08_anchor["mapi_10A_observed"]
    .fillna(______)
)

mapi_08_anchor["mapi_10A_value_source"] = np.select(
    [
        mapi_08_anchor["mapi_10A_observed"].notna(),
        (
            mapi_08_anchor["mapi_10A_observed"].isna()
            & mapi_08_anchor["ses-08A"].notna()
        ),
    ],
    [
        "observed_10A",
        "carried_from_08A",
    ],
    default="still_unavailable",
)

observed_10A_count = int(
    mapi_08_anchor["mapi_10A_observed"].notna().sum()
)
carried_08A_count = int(
    mapi_08_anchor["mapi_10A_value_source"]
    .eq("carried_from_08A")
    .sum()
)

locf_comparison = pd.DataFrame({
    "handling_decision": [
        "Observed session-10 values only",
        "Observed session-10 or LOCF from session 08",
    ],
    "session_08_anchor_cohort": [
        len(mapi_08_anchor),
        len(mapi_08_anchor),
    ],
    "session_10_analysis_values": [
        observed_10A_count,
        int(mapi_08_anchor["mapi_10A_locf_from_08A"].notna().sum()),
    ],
    "observed_session_10_values": [
        observed_10A_count,
        observed_10A_count,
    ],
    "carried_from_session_08": [
        0,
        carried_08A_count,
    ],
    "still_unavailable": [
        int(mapi_08_anchor["mapi_10A_observed"].isna().sum()),
        int(mapi_08_anchor["mapi_10A_locf_from_08A"].isna().sum()),
    ],
    "mean_session_10": [
        mapi_08_anchor["mapi_10A_observed"].mean(),
        mapi_08_anchor["mapi_10A_locf_from_08A"].mean(),
    ],
    "median_session_10": [
        mapi_08_anchor["mapi_10A_observed"].median(),
        mapi_08_anchor["mapi_10A_locf_from_08A"].median(),
    ],
})

locf_comparison[["mean_session_10", "median_session_10"]] = (
    locf_comparison[["mean_session_10", "median_session_10"]]
    .round(2)
)

display(
    mapi_08_anchor["mapi_10A_value_source"]
    .value_counts(dropna=False)
)
display(locf_comparison)

locf_plot = (
    locf_comparison
    .set_index("handling_decision")
    [["observed_session_10_values", "carried_from_session_08"]]
)

locf_plot.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 4),
)
plt.title("Observed and carried-forward session-10 analysis values")
plt.xlabel("Handling decision")
plt.ylabel("Number of analysis values")
plt.xticks(rotation=15, ha="right")
plt.legend(["Observed at session 10", "Carried from session 08"])
plt.tight_layout()
plt.show()

### Prepare for the Canvas reflection: LOCF sensitivity analysis

In three or four sentences:

1. Compare the number of observed session-10 values with the number of session-10 analysis values after LOCF.
2. State how many values were carried forward and identify the assumption that made those constructed values possible.
3. Explain why similar observed-only and LOCF means would not validate the LOCF assumption.
4. State what LOCF does not recover about the participants' actual session-10 MAPI scores.

These questions are assessed in Canvas; no additional notebook response is required.

## 4. Select and inspect the self-report variable

We now return to the central concordance question. Before joining self-report and hair toxicology, we must select a self-report variable using documentation and confirm that its observed values support a Yes/No classification.

### 4.1 Documentation-based selection

We will use:

`su_y_sui__use__mj__puff_001__l`

This variable directly records whether the participant reported using a cannabis product since the previous session. Its documented response options support the Yes/No classification needed for concordance.

We will not use:

`su_y_sui__branch__ud__mj_001__l`

The Data Dictionary identifies this as a Timeline Followback-derived field created for questionnaire branching and directs researchers to the original TLFB table for analysis. A variable can contain meaningful values and still be inappropriate for a research analysis because of how and why it was created.

The selected self-report item and hair toxicology do not cover identical periods. The comparison therefore describes agreement between two measures, not the accuracy of one measure against a perfect standard.

Data Dictionary:  
https://abcd.deapscience.com/#/my-datasets/create-dataset

Substance Use documentation:  
https://docs.abcdstudy.org/latest/documentation/non_imaging/su.html#substance-use-interview

In [ ]:
self_report_col = "su_y_sui__use__mj__puff_001__l"

sui_qc = su_y_sui[
    ["participant_id", "session_id", self_report_col]
].copy()

sui_qc["self_report_value"] = (
    sui_qc[self_report_col]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .fillna("Missing")
)

print("Selected self-report values, including missing:")
display(
    sui_qc["self_report_value"]
    .value_counts(dropna=False)
)

self_report_by_session = pd.crosstab(
    sui_qc["session_id"],
    sui_qc["self_report_value"],
    dropna=False,
).reindex(session_order, fill_value=0)

display(self_report_by_session)

expected_self_report_values = {"Yes", "No", "Missing"}
unexpected_self_report_values = sorted(
    set(sui_qc["self_report_value"].unique())
    - expected_self_report_values
)

print("Unexpected selected self-report values:")
display(unexpected_self_report_values)

## 5. Inspect hair toxicology availability

Exercise 3 traced the path from a scheduled in-person collection opportunity to a usable focal toxicology result. Lab 3 activates only the distinctions needed for the concordance analysis:

- collection, laboratory selection, and a usable focal result are different stages;
- laboratory testing was selective, so usable results do not represent all participant-sessions;
- code `666` identifies insufficient quantity for the focal THCCOOH result; and
- only `Positive` and `Negative` are interpretable for the binary concordance question.

A blank result, code `666`, a Negative result, and the absence of a hair row must remain distinct. Neither a Positive nor a Negative confirmation is ground truth.

Documentation:  
https://docs.abcdstudy.org/latest/documentation/non_imaging/su.html#hair-drug-toxicology

In [ ]:
hair_collection_col = "su_y_hairtox__coll_dtt"
hair_cnf_col = "su_y_hairtox__rslt__mj__thccooh_cnf"

hair_availability = su_y_hairtox[
    [
        "participant_id",
        "session_id",
        hair_collection_col,
        hair_cnf_col,
    ]
].copy()

hair_cnf_clean = (
    hair_availability[hair_cnf_col]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

hair_cnf_casefold = hair_cnf_clean.str.casefold()

hair_availability["hair_result_status"] = "other_or_unexpected"
hair_availability.loc[
    hair_cnf_clean.isna(),
    "hair_result_status",
] = "unavailable_result"
hair_availability.loc[
    hair_cnf_casefold.eq("positive").fillna(False),
    "hair_result_status",
] = "Positive"
hair_availability.loc[
    hair_cnf_casefold.eq("negative").fillna(False),
    "hair_result_status",
] = "Negative"
hair_availability.loc[
    hair_cnf_clean.isin(["666", "666.0"]),
    "hair_result_status",
] = "insufficient_quantity_666"

hair_result_by_session = pd.crosstab(
    hair_availability["session_id"],
    hair_availability["hair_result_status"],
    dropna=False,
).reindex(session_order, fill_value=0)

display(hair_result_by_session)

print("Hair-result states among rows present in the hair table:")
display(
    hair_availability["hair_result_status"]
    .value_counts(dropna=False)
)

### What to notice

The hair table describes only participant-sessions with a hair-table row. It cannot reveal participant-sessions absent from that table; the left join will identify those as `no_hair_row`.

A blank confirmation remains `unavailable_result`. Code `666` is retained as `insufficient_quantity_666`, not recoded as Negative or treated as a generic unexpected value. Only Positive and Negative records can enter the binary concordance denominator.

Because collection and testing were selective, the final comparison will describe a selected subset rather than substance-use prevalence in the cohort.

### Prepare for the Canvas reflection: measure selection and hair-result availability

Using the self-report and hair-toxicology outputs, respond in four or five sentences:

1. Why is `su_y_sui__use__mj__puff_001__l` more appropriate for the concordance analysis than `su_y_sui__branch__ud__mj_001__l`?
2. Report the observed counts of `Yes` and `No` for `su_y_sui__use__mj__puff_001__l`. Were any unexpected values found?
3. Among rows present in `su_y_hairtox`, report the counts of `Positive`, `Negative`, blank confirmation, and code `666`.
4. Why must a blank confirmation and code `666` remain distinct from a `Negative` result?
5. Why does this output not yet tell us how many self-report rows have no matching hair row?

These questions are assessed in Canvas; no additional notebook response is required.

## 6. Plan, validate, and perform the join

A **composite key** uses more than one column to identify a row. Here, `participant_id` and `session_id` jointly identify a participant-session.

**Cardinality** describes how rows in one table relate to rows in another. We expect at most one SUI row and one hair toxicology row for each participant-session in these extracts.

### Join plan

| Decision | Choice |
|---|---|
| Left table | `su_y_sui` |
| Right table | `su_y_hairtox` |
| Row unit | Participant-session |
| Composite key | `participant_id` + `session_id` |
| Expected relationship | One-to-one in these extracts |
| Join type | Left join |
| Reason | Preserve every self-report row and distinguish no hair row from other unavailable hair states |

The join includes only source columns needed for the research question. The original DataFrames remain unchanged.

### Availability information produced by the join

A `left_only` record tells us that no matching hair row appears in this extract. It is not a Negative result. It also does not establish whether the underlying reason was noncollection, nonselection, an unscheduled procedure, or another process.

A matched hair row can still contain a blank confirmation or code `666`. The join therefore identifies one stage in the availability pathway but does not eliminate the need to inspect the focal result.

In [ ]:
join_keys = ["participant_id", "session_id"]

def summarize_join_keys(table, table_name):
    return {
        "table": table_name,
        "rows": len(table),
        "unique_participant_session_pairs": (
            table[join_keys].drop_duplicates().shape[0]
        ),
        "duplicate_key_rows": int(
            table.duplicated(subset=join_keys).sum()
        ),
        "rows_missing_a_join_key": int(
            table[join_keys].isna().any(axis=1).sum()
        ),
    }

join_key_summary = pd.DataFrame([
    summarize_join_keys(su_y_sui, "su_y_sui"),
    summarize_join_keys(su_y_hairtox, "su_y_hairtox"),
])

display(join_key_summary)

sui_for_join = su_y_sui[
    ["participant_id", "session_id", self_report_col]
].copy()

hairtox_for_join = su_y_hairtox[
    [
        "participant_id",
        "session_id",
        hair_collection_col,
        hair_cnf_col,
    ]
].copy()

analysis_df = sui_for_join.merge(
    hairtox_for_join,
    on=join_keys,
    how="left",
    indicator=True,
    validate="one_to_one",
)

print("Self-report rows before join:", len(sui_for_join))
print("Rows after join:", len(analysis_df))
print()
print("Join results:")
display(analysis_df["_merge"].value_counts(dropna=False))

display(analysis_df.head())

### Prepare for the Canvas reflection: joining participant-session tables

In three or four sentences:

1. Explain how the composite key corresponds to the participant-session row unit.
2. State whether the join preserved the number of self-report rows.
3. Explain why a `left_only` record is neither a Negative hair result nor enough evidence to assign a specific missingness code.
4. Explain how selective hair collection and testing limit what the final joined subset represents.

These questions are assessed in Canvas; no additional notebook response is required.

## 7. Preserve unavailable-value meanings

Recoding creates analysis categories without changing the source fields.

For self-report, the extract supports `Yes`, `No`, `unavailable`, and `other_or_unexpected`. Because the extract does not distinguish every reason for a blank, we do not assign unsupported labels such as `555`, `777`, `888`, or `999`.

For hair toxicology, we preserve:

| Analysis status | Meaning |
|---|---|
| `no_hair_row` | No matching hair toxicology row for that participant-session |
| `unavailable_result` | A hair row exists but confirmation is blank |
| `insufficient_quantity_666` | A hair row exists but quantity was insufficient for the focal result |
| `Positive` | Interpretable positive confirmation |
| `Negative` | Interpretable negative confirmation |
| `other_or_unexpected` | Another nonblank value requiring review |

In [ ]:
# Derive self-report status while preserving the source column.
raw_self_report = analysis_df[self_report_col].astype("string").str.strip()

analysis_df["self_report_status"] = "other_or_unexpected"
analysis_df.loc[
    raw_self_report.isna() | raw_self_report.eq(""),
    "self_report_status",
] = "unavailable"
analysis_df.loc[
    raw_self_report.str.casefold().eq("yes").fillna(False),
    "self_report_status",
] = "Yes"
analysis_df.loc[
    raw_self_report.str.casefold().eq("no").fillna(False),
    "self_report_status",
] = "No"

# Derive hair status using both join status and raw confirmation.
raw_hair_confirmation = (
    analysis_df[hair_cnf_col]
    .astype("string")
    .str.strip()
)
matched_hair_row = analysis_df["_merge"].eq("both")
hair_casefold = raw_hair_confirmation.str.casefold()

analysis_df["hair_status"] = "other_or_unexpected"
analysis_df.loc[
    analysis_df["_merge"].eq("left_only"),
    "hair_status",
] = "no_hair_row"
analysis_df.loc[
    matched_hair_row
    & (raw_hair_confirmation.isna() | raw_hair_confirmation.eq("")),
    "hair_status",
] = "unavailable_result"
analysis_df.loc[
    matched_hair_row
    & raw_hair_confirmation.isin(["666", "666.0"]),
    "hair_status",
] = "insufficient_quantity_666"
analysis_df.loc[
    matched_hair_row
    & hair_casefold.eq("positive").fillna(False),
    "hair_status",
] = "Positive"
analysis_df.loc[
    matched_hair_row
    & hair_casefold.eq("negative").fillna(False),
    "hair_status",
] = "Negative"

print("Self-report analysis categories:")
display(analysis_df["self_report_status"].value_counts(dropna=False))

print()
print("Hair toxicology analysis categories:")
display(analysis_df["hair_status"].value_counts(dropna=False))

### What to notice

The derived status fields preserve distinctions that would disappear if unavailable states were converted to zero, No, or Negative.

The notebook does not assign a more specific reason than the source records and documentation support. In particular, `no_hair_row`, a blank confirmation, and documented code `666` remain separate.

## 8. Create the concordance fields and audit the denominator

A **Boolean field** stores a True/False condition. An **analytic denominator** is the set of records eligible to contribute to a particular result.

For this analysis, a participant-session is eligible only when:

- self-report status is `Yes` or `No`; and
- hair status is `Positive` or `Negative`.

All other rows remain in `analysis_df`, but their concordance status is indeterminate.

### Your Turn 3: define valid concordance pairs

Complete the two matching method blanks in the next cell so a record is valid only when both classifications are available.

**Clue:** Use the pandas method that asks whether a value is not missing. The `&` operator requires both conditions to be true.

**Further learning with Copilot:** Ask: *Explain the difference between `isna()` and `notna()` in pandas. Then explain why two `notna()` conditions joined with `&` define records with both measures available. Do not complete my assignment code.*

In [ ]:
# Use nullable integer fields so unavailable values remain missing.
analysis_df["self_report_positive"] = pd.Series(
    pd.NA,
    index=analysis_df.index,
    dtype="Int64",
)
analysis_df.loc[
    analysis_df["self_report_status"].eq("Yes"),
    "self_report_positive",
] = 1
analysis_df.loc[
    analysis_df["self_report_status"].eq("No"),
    "self_report_positive",
] = 0

analysis_df["hair_positive"] = pd.Series(
    pd.NA,
    index=analysis_df.index,
    dtype="Int64",
)
analysis_df.loc[
    analysis_df["hair_status"].eq("Positive"),
    "hair_positive",
] = 1
analysis_df.loc[
    analysis_df["hair_status"].eq("Negative"),
    "hair_positive",
] = 0

analysis_df["valid_concordance_pair"] = (
    analysis_df["self_report_positive"].______()
    & analysis_df["hair_positive"].______()
)

analysis_df["concordance_status"] = "indeterminate"
analysis_df.loc[
    analysis_df["self_report_positive"].eq(1)
    & analysis_df["hair_positive"].eq(1),
    "concordance_status",
] = "concordant_positive"
analysis_df.loc[
    analysis_df["self_report_positive"].eq(0)
    & analysis_df["hair_positive"].eq(0),
    "concordance_status",
] = "concordant_negative"
analysis_df.loc[
    analysis_df["self_report_positive"].eq(1)
    & analysis_df["hair_positive"].eq(0),
    "concordance_status",
] = "self_yes_hair_negative"
analysis_df.loc[
    analysis_df["self_report_positive"].eq(0)
    & analysis_df["hair_positive"].eq(1),
    "concordance_status",
] = "self_no_hair_positive"

display(
    analysis_df[
        [
            "participant_id",
            "session_id",
            "self_report_status",
            "hair_status",
            "valid_concordance_pair",
            "concordance_status",
        ]
    ].head(12)
)

### Audit the denominator

The audit separates the major availability states. These rows are not all sequential filters, so the final row is the intersection of completed self-report and interpretable hair confirmation.

In [ ]:
denominator_audit = pd.DataFrame({
    "criterion": [
        "All joined self-report rows",
        "Completed self-report item: Yes or No",
        "Matching hair toxicology row",
        "Hair row with blank confirmation",
        "Hair row with insufficient quantity (666)",
        "Interpretable hair confirmation: Positive or Negative",
        "Valid concordance pair: both conditions",
    ],
    "rows": [
        len(analysis_df),
        int(analysis_df["self_report_status"].isin(["Yes", "No"]).sum()),
        int(analysis_df["_merge"].eq("both").sum()),
        int(analysis_df["hair_status"].eq("unavailable_result").sum()),
        int(analysis_df["hair_status"].eq("insufficient_quantity_666").sum()),
        int(analysis_df["hair_status"].isin(["Positive", "Negative"]).sum()),
        int(analysis_df["valid_concordance_pair"].sum()),
    ],
})

denominator_audit["percent_of_all_joined_rows"] = (
    denominator_audit["rows"] / len(analysis_df) * 100
).round(1)

display(denominator_audit)

### Prepare for the Canvas reflection: auditing the analytic denominator

Using the derived-status counts and denominator audit, respond in four or five sentences:

1. How many joined participant-session rows were present, and how many formed valid concordance pairs?
2. What percentage of all joined rows contributed to the concordance analysis?
3. Report the numbers excluded because there was no matching hair row, the confirmation was blank, or the result was code `666`.
4. State the two conditions required for `valid_concordance_pair` to be `True`.
5. Explain why recoding unavailable hair states as `Negative` would change both the denominator and the scientific interpretation.

These questions are assessed in Canvas; no additional notebook response is required.

## 9. Analyze categorical × categorical concordance

Lab 2 introduced a categorical × quantitative comparison. Lab 3 now introduces categorical × categorical bivariate analysis.

Because the row unit is a participant-session, one participant may contribute more than one valid record. The results describe participant-session records, not unique individuals.

We begin with a two-by-two contingency table for valid self-report Yes/No and hair Negative/Positive pairs. The table makes the four concordance and discordance cells visible before we summarize them.

In [ ]:
valid_df = analysis_df.loc[
    analysis_df["valid_concordance_pair"]
].copy()

self_report_order = ["No", "Yes"]
hair_order = ["Negative", "Positive"]

concordance_crosstab = pd.crosstab(
    pd.Categorical(
        valid_df["self_report_status"],
        categories=self_report_order,
        ordered=True,
    ),
    pd.Categorical(
        valid_df["hair_status"],
        categories=hair_order,
        ordered=True,
    ),
    rownames=["Self-report"],
    colnames=["Hair confirmation"],
    margins=True,
)

display(concordance_crosstab)

plot_crosstab = concordance_crosstab.loc[
    self_report_order,
    hair_order,
]

plot_crosstab.plot(
    kind="bar",
    figsize=(7, 4),
)
plt.title("Hair confirmation by self-report status")
plt.xlabel("Self-report status")
plt.ylabel("Participant-session records")
plt.xticks(rotation=0)
plt.legend(title="Hair confirmation")
plt.tight_layout()
plt.show()

### From the crosstab to four interpretation categories

The four interior cells map directly to the concordance categories:

| Self-report | Hair | Category |
|---|---|---|
| Yes | Positive | `concordant_positive` |
| No | Negative | `concordant_negative` |
| Yes | Negative | `self_yes_hair_negative` |
| No | Positive | `self_no_hair_positive` |

Concordance is not proof that both measures are correct. Discordance is not proof that a participant lied. The measures have different reference periods and limitations, and hair testing was selective.

In [ ]:
concordance_order = [
    "concordant_positive",
    "concordant_negative",
    "self_yes_hair_negative",
    "self_no_hair_positive",
]

concordance_counts = (
    valid_df["concordance_status"]
    .value_counts()
    .reindex(concordance_order, fill_value=0)
)

concordance_results = pd.DataFrame({
    "concordance_status": concordance_counts.index,
    "count": concordance_counts.values,
})
concordance_results["percent_of_valid_pairs"] = (
    concordance_results["count"] / len(valid_df) * 100
).round(2)

print("Valid participant-session denominator:", len(valid_df))
display(concordance_results)

print(
    "Crosstab interior total:",
    int(plot_crosstab.to_numpy().sum()),
)

### Prepare for the Canvas reflection: interpreting concordance

Write four or five sentences that:

1. name the valid participant-session denominator;
2. identify the most common concordance or discordance category;
3. report both directions of disagreement separately;
4. explain why discordance is not evidence that a participant lied and why hair confirmation is not ground truth; and
5. explain why the results apply only to participant-sessions with two valid measures and should not be generalized to the full cohort.

Your interpretation may consider reference-period differences, detection windows, reporting context, timing, selective hair testing, sample availability, and assay limitations.

These questions are assessed in Canvas; no additional notebook response is required.

## 10. Document the cleaning, derivation, and imputation decisions

A **cleaning log** is a provenance record. It states what issue was encountered, what action was taken, why the action was justified, which documentation informed it, and how it affected the analysis.

The log must distinguish observed source values from constructed analysis values.

In [ ]:
cleaning_log = pd.DataFrame([
    {
        "variable_or_field": "MAPI items 013-015",
        "issue": "Entirely blank and outside the documented 15-item score",
        "action": "Exclude from score inputs; do not zero-fill",
        "justification": "The official score uses items 001-012 and 016-018",
        "documentation_source": "ABCDscores MAPI scoring documentation",
        "analytic_consequence": "Score architecture and eligible item denominator",
    },
    {
        "variable_or_field": "mapi_10A_locf_from_08A",
        "issue": "Some session-08 anchor participants lack an observed session-10 score",
        "action": "Create a separate LOCF sensitivity value without changing the observed column",
        "justification": "Exercise 3 comparison rule; assumes no score change from session 08 to 10",
        "documentation_source": "Exercise 3 and the Lab 3 LOCF analysis plan",
        "analytic_consequence": "Increases analysis values without increasing observed session-10 evidence",
    },
    {
        "variable_or_field": hair_cnf_col,
        "issue": "A matching hair row can have a blank confirmation",
        "action": "Derive unavailable_result and exclude it from the binary comparison",
        "justification": "A blank confirmation is not an observed Negative result",
        "documentation_source": "ABCD hair toxicology documentation and Data Dictionary",
        "analytic_consequence": "Concordance denominator",
    },
    {
        "variable_or_field": hair_cnf_col,
        "issue": "Code 666 records insufficient quantity for the focal result",
        "action": "Preserve as insufficient_quantity_666",
        "justification": "Insufficient quantity is distinct from blank, Negative, and no hair row",
        "documentation_source": "Exercise 3 Model 2 and ABCD curation documentation",
        "analytic_consequence": "Availability classification and concordance denominator",
    },
    {
        "variable_or_field": "_merge",
        "issue": "No matching hair row after the left join",
        "action": "Derive no_hair_row and retain the self-report row",
        "justification": "Absence of a hair record is not a Negative test",
        "documentation_source": "Join plan and participant-session key validation",
        "analytic_consequence": "Availability classification and concordance denominator",
    },
    {
        "variable_or_field": "valid_concordance_pair",
        "issue": "Not every joined row has two interpretable classifications",
        "action": "Include only Yes/No self-report with Positive/Negative hair confirmation",
        "justification": "Concordance requires two defined classifications",
        "documentation_source": "Documented analysis plan in this notebook",
        "analytic_consequence": "Final participant-session denominator",
    },
])

display(cleaning_log)

### Your Turn 4: add a cleaning-log row

Complete one additional row for a decision that should be documented. Choose a decision not already fully represented in the log, and state its consequence for the analysis.

In [ ]:
your_cleaning_log_row = pd.DataFrame({
    "variable_or_field": ["TODO: variable or derived field"],
    "issue": ["TODO: describe the issue"],
    "action": ["TODO: describe the action"],
    "justification": ["TODO: explain why the action is appropriate"],
    "documentation_source": ["TODO: cite the data dictionary, codebook, or notebook rule"],
    "analytic_consequence": ["TODO: explain what part of the analysis is affected"],
})

completed_cleaning_log = pd.concat(
    [cleaning_log, your_cleaning_log_row],
    ignore_index=True,
)

display(completed_cleaning_log)

### Prepare for the Canvas reflection: documenting a cleaning decision

Using the cleaning-log row you added, respond in three or four sentences:

1. Which variable or derived field did you document, and what issue did it address?
2. Why was your action justified by documentation or an explicitly stated notebook rule rather than convenience?
3. What part of the analysis would be different if you had made a different decision?

This question is assessed in Canvas; no additional notebook response is required.

## Finish the lab and complete the Canvas reflection

Before completing the Canvas reflection quiz:

1. Complete all four **Your Turn** activities.
2. Review all seven Canvas reflection-preparation prompts and locate the outputs or cleaning-log entry needed to answer them.
3. Run all cells from top to bottom so the outputs are current.
4. Confirm that source DataFrames and observed source columns remain unchanged.
5. Verify that observed and carried-forward session-10 values remain distinguishable.
6. Confirm that the crosstab and four-category results use the same valid participant-session denominator.
7. Save the notebook for your own continued work; the notebook itself is not collected.

Use the displayed outputs to complete the corresponding Canvas quiz. It assesses unavailable-data codes, observed versus imputed values, complete-case versus LOCF logic, denominator selection, crosstab interpretation, the limits of concordance claims, and documentation of cleaning decisions.